# Yoruba (New Testament)

In [ ]:
"""
Optimized synthesis script for EveryVoice.
Loads models once and synthesizes all sentences in batches.
"""

import os
from pathlib import Path
import soundfile as sf
import io

import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Audio

# ---- Import EveryVoice internals ----
from everyvoice.model.feature_prediction.FastSpeech2_lightning.fs2.model import FastSpeech2
from everyvoice.model.feature_prediction.FastSpeech2_lightning.fs2.cli.synthesize import (
    synthesize_helper,
    get_global_step,
)
from everyvoice.model.feature_prediction.FastSpeech2_lightning.fs2.type_definitions import (
    SynthesizeOutputFormats,
)
from everyvoice.config.type_definitions import DatasetTextRepresentation
from everyvoice.model.vocoder.HiFiGAN_iSTFT_lightning.hfgl.utils import (
    load_hifigan_from_checkpoint,
)
from everyvoice.utils.heavy import get_device_from_accelerator

# ---- Load models ONCE ----
device = get_device_from_accelerator("gpu")
print(f"Using device: {device}")

In [ ]:
LANGUAGE = "Yoruba"

CKPT_PATH = "Open-Bible-Yoruba-NT/logs_and_checkpoints/FeaturePredictionExperiment/base/checkpoints/last.ckpt"
VOCODER_CKPT_PATH = "/home/mila/g/guzmand/scratch/checkpoints/hifigan_universal_v1_everyvoice.ckpt"
OUTPUT_DIR = "synthesis_output/yoruba-nt"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Load test set ──────────────────────────────────────────────────────────────
def get_audio_filename(example):
    # Now example["audio"] is {'path': '...', 'bytes': b'...'}
    example["filename"] = example["audio"]["path"].split("/")[-1]
    return example
    
ds = load_dataset(
    "parquet",
    data_files={"test": f"hf://datasets/davidguzmanr/open-bible-resources/{LANGUAGE}/test-*.parquet"},
    split="test"
)
ds = ds.cast_column("audio", Audio(decode=False))
ds = ds.map(get_audio_filename)

# Convert to pandas DataFrame and remove the audio column
test = ds.remove_columns("audio").to_pandas()
print(f"Validation samples: {len(test)}")

test_csv_path = os.path.join(OUTPUT_DIR, "test.csv")
test.to_csv(test_csv_path, index=False)
print(f"Saved test dataframe to: {test_csv_path}")

test.head()

In [ ]:
# # ── Save ground-truth audio files ─────────────────────────────────────────────
GROUND_TRUTH_DIR = f"{OUTPUT_DIR}/ground-truth"
os.makedirs(GROUND_TRUTH_DIR, exist_ok=True)

for example in tqdm(ds, desc="Saving ground-truth WAVs"):
    filename = example["filename"]                  # e.g. "abc123.mp3"
    stem     = os.path.splitext(filename)[0]        # strip original extension
    out_path = f"{GROUND_TRUTH_DIR}/{stem}.wav"

    audio_bytes = example["audio"]["bytes"]         # raw bytes from the parquet
    
    # Read from bytes buffer → numpy array + sample rate
    with io.BytesIO(audio_bytes) as buf:
        audio_array, sample_rate = sf.read(buf)

    sf.write(out_path, audio_array, sample_rate)

print(f"Saved {len(ds)} WAV files to {GROUND_TRUTH_DIR}")

In [ ]:
test = pd.read_csv(test_csv_path, sep='\t')
test

In [ ]:
# ---- Configuration ----
feature_prediction_checkpoint = Path(CKPT_PATH)
vocoder_base_checkpoint = Path(VOCODER_CKPT_PATH)
output_dir = Path(OUTPUT_DIR)
wav_dir = output_dir / "wav"

# ---- Read test data ----
test = pd.read_csv(test_csv_path, sep=',')
test = test.rename(columns={'filename': 'file'})
test = test[["file", "text"]]
print(f"Total sentences to synthesize: {len(test)}")

In [ ]:
test = test.head(5)

In [ ]:

print("Loading feature prediction model...")
model = FastSpeech2.load_from_checkpoint(str(feature_prediction_checkpoint)).to(device)
model.eval()
global_step = get_global_step(feature_prediction_checkpoint)

print("Loading vocoder...")
vocoder_ckpt = torch.load(str(vocoder_base_checkpoint), map_location=device, weights_only=True)
vocoder_model, vocoder_config = load_hifigan_from_checkpoint(vocoder_ckpt, device)
vocoder_global_step = get_global_step(vocoder_base_checkpoint)

print("Models loaded successfully!")

# ---- Prepare data with custom basenames ----
DEFAULT_LANGUAGE = next(iter(model.lang2id.keys()), None)
DEFAULT_SPEAKER = next(iter(model.speaker2id.keys()), None)

filelist_data = []
for _, row in test.iterrows():
    basename = os.path.splitext(row["file"])[0]
    filelist_data.append({
        "basename": basename,
        "characters": row["text"],
        "language": DEFAULT_LANGUAGE,
        "speaker": DEFAULT_SPEAKER,
        "duration_control": 1.0,
    })

print(f"Prepared {len(filelist_data)} entries for synthesis")

# ---- Run batch synthesis ----
print("\nStarting batch synthesis...")

synthesize_helper(
    model=model,
    texts=None,
    style_reference=None,
    language=None,
    speaker=None,
    duration_control=1.0,
    global_step=global_step,
    output_type=[SynthesizeOutputFormats.wav],
    text_representation=DatasetTextRepresentation.characters,
    accelerator="gpu",
    devices="auto",
    device=device,
    batch_size=4,
    num_workers=4,
    filelist=None,
    filelist_data=filelist_data,
    output_dir=output_dir,
    teacher_forcing_directory=None,
    vocoder_model=vocoder_model,
    vocoder_config=vocoder_config,
    vocoder_global_step=vocoder_global_step,
)

print("Batch synthesis complete!")

# ---- Rename output files ----
# EveryVoice names outputs as: basename--speaker--language--ckpt=N--v_ckpt=M--pred.wav
# We rename them to just: basename.wav
wav_files = list(wav_dir.glob("*.wav")) if wav_dir.exists() else []
print(f"Found {len(wav_files)} generated wav files")

renamed = 0
for wav_path in wav_files:
    parts = wav_path.name.split("--")
    original_basename = parts[0]
    target_name = f"{original_basename}.wav"
    target_path = wav_dir / target_name

    if wav_path.name != target_name:
        os.rename(wav_path, target_path)
        renamed += 1

print(f"Renamed {renamed} files")

# Verify all expected files exist
missing = []
for _, row in test.iterrows():
    expected_path = wav_dir / row["file"]
    if not expected_path.exists():
        missing.append(row["file"])

if missing:
    print(f"\nMissing {len(missing)} files:")
    for f in missing[:10]:
        print(f"  - {f}")
    if len(missing) > 10:
        print(f"  ... and {len(missing) - 10} more")
else:
    print(f"\nAll {len(test)} expected wav files are present!")


In [ ]:
# ── Quick sanity check: listen to a sample ─────────────────────────────────────
import IPython.display as ipd

ref_index = 0

stem = os.path.splitext(test.iloc[ref_index]["file"])[0]
sample_path = wav_dir / f"{stem}.wav"
gt_path = os.path.join(GROUND_TRUTH_DIR, f"{stem}.wav")
transcript = test.iloc[ref_index]["text"]

ipd.display(ipd.Markdown(f"""
**Transcript:** {transcript}

**Generated audio:**
"""))
ipd.display(ipd.Audio(sample_path))

ipd.display(ipd.Markdown("**Ground-truth audio:**"))
ipd.display(ipd.Audio(gt_path))

In [ ]:
# ── Quick sanity check: listen to a sample ─────────────────────────────────────
import os
from pathlib import Path
import pandas as pd
import IPython.display as ipd

GROUND_TRUTH_DIR = f"{OUTPUT_DIR}/ground-truth"
wav_dir = Path(OUTPUT_DIR) / "wav"

test_csv_path = os.path.join(OUTPUT_DIR, "test.csv")
test = pd.read_csv(test_csv_path)
test = test.rename(columns={"filename": "file"})

ref_index = 0

stem = os.path.splitext(test.iloc[ref_index]["file"])[0]
sample_path = wav_dir / f"{stem}.wav"
gt_path = os.path.join(GROUND_TRUTH_DIR, f"{stem}.wav")
transcript = test.iloc[ref_index]["text"]

ipd.display(ipd.Markdown(f"""
**Transcript:** {transcript}

**Generated audio:**
"""))
ipd.display(ipd.Audio(sample_path))

ipd.display(ipd.Markdown("**Ground-truth audio:**"))
ipd.display(ipd.Audio(gt_path))